# L-FTS with Lamellar Phase

<!-- This example demonstrates how to find the saddle-point solution of AB diblock copolymer systems with Anderson Mixing. -->
* Saddle-point solutions for imaginary fields are found efficiently using Anderson mixing or linear response theory.

* The real fields are updated using the Leimkuhler-Matthews method:
\begin{align}
\Omega_i({\bf r}, \tau+\delta\tau) &= \Omega_i({\bf r}, \tau) -λ_i\frac{\beta}{\rho_0}\frac{\delta H[\{\Omega_i\},\{w_j\}]}{\delta \Omega_i({\bf r}, \tau)} \delta\tau + \frac{\sigma_i}{2}\left(\mathcal{N}({\bf r}, \tau)+\mathcal{N}({\bf r}, \tau+\delta\tau)\right),  \ \ \ \ (\textrm{per monomer unit}) \\
\end{align}
where $\mathcal{N}$ is Gaussian noise and $\sigma_i^2 = \lambda_i\frac{2\delta\tau N^2}{\Delta V \rho_0}$.

* This example is based on the equations in `PolymerFieldTheory/IncompressibleModel.ipynb` and uses the `symmetric_formulation.py` and `lfts.py` modules.

References:
* [(2009) Fast and accurate SCFT calculations for periodic block-copolymer morphologies using the spectral method with Anderson mixing](http://dx.doi.org/10.1140/epje/i2009-10534-3)
* [(2011) Efficiency of pseudo-spectral algorithms with Anderson mixing for the SCFT of periodic block-copolymer phases](http://dx.doi.org/10.1140/epje/i2011-11110-0)
* [(2021) Fluctuation correction for the order–disorder transition of diblock copolymer melts](https://doi.org/10.1063/5.0046167)
* [(2023) Efficient Langevin and Monte Carlo sampling algorithms: The case of field-theoretic simulations](https://doi.org/10.1063/5.0131183)
* [(2025) Improved Algorithm for Identifying Partial Saddle-Points in Polymer Field Theoretic Simulations](https://doi.org/10.1063/5.0264036)

### 1. Setting simulation parameters

#### Static simulation parameters

| Parameter | Symbol | Value |
|:----------|:------:|------:|
| Reference chain length | $N_\text{Ref}$ | 90 |
| Contour step | $\Delta s$ | 1/90 |
| Reference length | $R_0$ | $b N_\text{Ref}^{1/2}$ |
| Box size X | $L_x$ | 4.36 $R_0$ |
| Box size Y | $L_y$ | 4.36 $R_0$ |
| Box size Z | $L_z$ | 4.36 $R_0$ |
| Grid points X | $m_x$ | 40 |
| Grid points Y | $m_y$ | 40 |
| Grid points Z | $m_z$ | 40 |
| Segment length A | $b_A/b$ | 1.0 |
| Segment length B | $b_B/b$ | 1.0 |
| Interaction parameter | $\chi N$ | 17 |
| A-block fraction | $f_A$ | 0.4 |

#### Langevin dynamics parameters

| Parameter | Symbol | Value |
|:----------|:------:|------:|
| Invariant polymerization index | $\bar{N}$ | 10000 |
| Time step | $\delta\tau N$ | 8.0 |
| Number of steps | - | 200 |

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"      # Single-threaded OpenMP

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from polymerfts import lfts

f = 0.4        # A-fraction of major BCP chain, f

params = {
    #---------------- Simulation parameters -----------------------------
    "nx":[40, 40, 40],          # Simulation grid numbers
    "lx":[4.36, 4.36, 4.36],    # Simulation box size as a_Ref * N_Ref^(1/2) unit,
                                # where "a_Ref" is reference statistical segment length
                                # and "N_Ref" is the number of segments of reference linear homopolymer chain.

    "chain_model":"discrete",   # "discrete" or "continuous" chain model
    "ds":1/90,                  # Contour step interval, which is equal to 1/N_Ref.

    "segment_lengths":{         # Relative statistical segment length compared to "a_Ref.
        "A":1.0, 
        "B":1.0, },

    "chi_n": {"A,B":17.0},     # Bare interaction parameter, Flory-Huggins params * N_Ref

    "polymers":[{      # Polymer species
        "volume_fraction":1.0,  # Volume fraction of polymer chain
        "blocks":[              # AB diBlock Copolymer
            {"type":"A", "length":f, }, # A-block
            {"type":"B", "length":1-f}, # B-block
        ],},],
        
    "langevin":{                # Langevin Dynamics
        "max_step":200,         # Langevin steps for simulation
        "dt":8.0,               # Langevin step interval, delta tau*N_Ref
        "nbar":10000,           # Invariant polymerization index, nbar of N_Ref
    },
    
    "recording":{                             # Recording Simulation Data
        "dir":"data_simulation",              # Directory name
        "checkpoint_interval":100, # 10000       # Period for recording concentrations and fields
        "sample_interval":10,             # Period for computing structure function
        "statistics_save_interval":100, # 100000   # Period for recording structure function
    },

    "saddle":{                # Iteration for the pressure field 
        "max_iter" :100,      # Maximum number of iterations
        "tolerance":1e-4,     # Tolerance of incompressibility 
    },

    "compressor":{
        # "name":"am",                # Anderson Mixing
        # "name":"lr",                # Linear Response
        "name":"lram",              # Linear Response + Anderson Mixing
        "max_hist":20,              # Maximum number of history
        "start_error":5e-1,         # When switch to AM from simple mixing
        "mix_min":0.01,             # Minimum mixing rate of simple mixing
        "mix_init":0.01,            # Initial mixing rate of simple mixing
    },

    "verbose_level":1,      # 1 : Print at each langevin step.
                            # 2 : Print at each saddle point iteration.
}

### 2. Initial potential fields

In [ ]:
nx = params["nx"]
lx = params["lx"]

# Set initial fields
input_data = loadmat("LamellaInput.mat", squeeze_me=True)
w_A = np.reshape(input_data["w_A"], nx)
w_B = np.reshape(input_data["w_B"], nx)

w = {"A": w_A, "B": w_B}  # potential field dictionary

# Plot each image with the same vmin and vmax
vmin = min(w_A.min(), w_B.min())
vmax = max(w_A.max(), w_B.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Initial Potential Fields")
im = axes[0].imshow(w_A[0,:,:], extent=(0, lx[1], 0, lx[0]), origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
im = axes[1].imshow(w_B[0,:,:], extent=(0, lx[1], 0, lx[0]), origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set(title='$w_A$', xlabel='y', ylabel='x')
axes[1].set(title='$w_B$', xlabel='y', ylabel='x')

fig.subplots_adjust(right=0.92)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
fig.show()

### 3. Initialization

In [ ]:
# Set random seed
# If you want to obtain different results for each execution, set random_seed=None
random_seed = 12345

# Initialize simulation
simulation = lfts.LFTS(params=params, random_seed=random_seed)

### 4. Run

In [ ]:
# Run
simulation.run(initial_fields={"A": w_A, "B": w_B})

### 5. Display $\phi_A({\bf r})$

1. Use `plot_3d_isodensity.ipynb` in `scripts/analysis/`.

In [ ]:
import plotly.graph_objects as go

mdic = loadmat("data_simulation/fields_000100.mat", squeeze_me=True)
nx = mdic['nx']
lx = mdic['lx']
phi_a = mdic['phi_A']

X, Y, Z = np.mgrid[0:nx[0], 0:nx[1], 0:nx[2]]
values = np.reshape(phi_a, nx)

fig = go.Figure(data=go.Isosurface(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=values.flatten(),
    isomin=0.0,
    isomax=1.0,
    opacity=0.2, 
    surface_count=9, # number of isosurface. 2 by default:only min and max
    #caps=dict(x_show=False, y_show=False, z_show=False),
    colorscale='RdBu', #RdBu, jet, Plotly3, OrRd
    colorbar=dict(thickness=30,tickfont=dict(size=30,color="black")),
    reversescale=True,
    ))
fig.update_layout(
    autosize=False,
    width=500,
    height=500,
)
fig.update_scenes(
    camera_projection_type="orthographic",
    xaxis_visible=False,
    yaxis_visible=False,
    zaxis_visible=False)

### 6. Display structure function $S(k)$

1. The structure functions are defined as:
\begin{align}
S_{ij}({\bf k})&= \frac{1}{V} \int d{\bf r} \int d{\bf r'} e^{−i{\bf k}\cdot({\bf r}−{\bf r}')}\left<\delta\hat{\rho}_i({\bf r})\delta\hat{\rho}_j({\bf r}')\right>, \\
\end{align}
where $\delta\hat{\rho}_i({\bf r})= \hat{\rho}_i({\bf r}) - \left<\hat{\rho}_i({\bf r})\right>$ and $\hat{\rho}_i({\bf r})/\rho_0 = \hat{\phi}_i({\bf r})$.

2. To plot the angle-averaged $S(k)$ of this run, copy [`scripts/analysis/plot_structure_function.ipynb`](../../scripts/analysis/plot_structure_function.ipynb) into this folder and set `file_path = 'data_simulation/structure_function'`, `type_pair = ("A", "A")` (or `("A", "B")`, `("B", "B")`) and `iterations = range(100, 201, 100)`. It groups the Fourier modes by $|{\bf k}|$, weights the half spectrum and the files by their numbers of samples, and also handles cells that are not orthogonal.

References:
* [(2014) A multi-species exchange model for fully fluctuating polymer field theory simulations](https://aip.scitation.org/doi/pdf/10.1063/1.4900574)
* [(2023) Field-Theoretic Simulations in Soft Matter and Quantum Fluids](https://doi.org/10.1093/oso/9780192847485.001.0001)
* [(2025) Polymer Field Theory for Multimonomer Incompressible Models: Symmetric Formulation and ABC Systems](https://doi.org/10.1021/acs.macromol.4c02636)